In [ ]:
# conda activate genomic_tools

import os
import json
import pickle
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', None)

## Load GTF

## Load interproscan results

In [ ]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [ ]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [ ]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label'] 

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [ ]:
def get_skip_junction_aa(skip_transcript, exon_cds_start, exon_cds_end, cds_by_transcript):
    cds = _cds_rows(cds_by_transcript.get(skip_transcript))
    if cds is None:
        return None
    nt_before = 0
    for c in cds:
        # stop when we reach or pass the cassette exon's position
        if c['start'] >= exon_cds_start:
            break
        # only count rows that end before the cassette exon starts
        if c['end'] < exon_cds_start:
            nt_before += c['end'] - c['start'] + 1
        else:
            # partial overlap: count only up to exon_cds_start
            nt_before += exon_cds_start - c['start']
            break
    return nt_before // 3

def near_junction(df, junction_aa, window=50):
    """Keep features within window aa of junction_aa on either side."""
    return df[
        (df['stop'] >= junction_aa - window) &
        (df['start'] <= junction_aa + window)
    ]
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

In [ ]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [ ]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [ ]:
columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD']
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)

for ev, rec in event_protein_map.items():

    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[
        (incl_df['start'] <= rec_incl['aa_end']) &
        (incl_df['stop']  >= rec_incl['aa_start'])
    ]
    if overlap_df.empty:
        continue
    event_interproscan_map[ev]['inclusion'] = overlap_df.assign(
        aa_start = rec_incl['aa_start'],
        aa_end = rec_incl['aa_end'],
        exon_cds_start = rec_incl['exon_cds_start'],
        exon_cds_end = rec_incl['exon_cds_end'],
        frame_preserving = rec_incl['frame_preserving'],
        clean_start = rec_incl['clean_start'],
        clean_end = rec_incl['clean_end'],
    ).reset_index(drop=True)

    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None:
            junction_aa = get_skip_junction_aa(
                rec['real_skip'], rec_incl['exon_cds_start'],
                rec_incl['exon_cds_end'], cds_by_transcript
            ) or rec_incl['aa_start']
            s = near_junction(skip_df, junction_aa).assign(
                truncation_aa = junction_aa,
                frame_preserving = rec_incl['frame_preserving'],
                aa_start = junction_aa,   # reference point for coordinate conversion
                aa_end = junction_aa,
                exon_cds_start   = rec_incl['exon_cds_start'],
                exon_cds_end = rec_incl['exon_cds_end'],
            )
            if not s.empty:
                event_interproscan_map[ev]['real_skip'] = s.reset_index(drop=True)

    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            junction_aa = rec_incl['aa_start']
            s = near_junction(synth_df, junction_aa).assign(
                truncation_aa = junction_aa,
                frame_preserving = rec_incl['frame_preserving'],
                aa_start = junction_aa,   # reference point for coordinate conversion
                aa_end = junction_aa,
                exon_cds_start = rec_incl['exon_cds_start'],
                exon_cds_end = rec_incl['exon_cds_end'],
            )
            if not s.empty:
                event_interproscan_map[ev]['synthetic_skip'] = s.reset_index(drop=True)

    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        frames = []
        for sib in rec['exon_diff_junction_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(
                frames, ignore_index=True
            )

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        frames = []
        for sib in rec['exon_diff_boundary_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(
                frames, ignore_index=True
            )

In [ ]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [ ]:
with open("data/event_interproscan_map.pkl", "rb") as file:
    event_interproscan_map = pickle.load(file)

In [ ]:
# merge cell type-specific events with InterProScan results

signif_event_interproscan_map = dict() 
signif_event_interproscan_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: event_interproscan_map[ev] for ev in signif_events_df.index 
            if ev in event_interproscan_map
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        signif_event_interproscan_map[ctype] = df
        
        # summarize interpro results for significant splicing events
        signif_event_interproscan_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_start=('exon_cds_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_end=('exon_cds_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [ ]:
with open("data/signif_event_interproscan_map.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_map, file)
    
with open("data/signif_event_interproscan_summary.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_summary, file)

Preview

In [ ]:
rec = signif_event_interproscan_summary['Deep_layer_glutamatergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

NameError: name 'signif_event_interproscan_summary' is not defined